## Part 1 - Data Conditioning

### Import Libraries

In [1]:
import polars as pl

In [2]:
# Set the streaming engine as the default for all future operations
pl.Config.set_engine_affinity("streaming")
# Show all rows
pl.Config.set_tbl_rows(-1)

polars.config.Config

### Load Data

In [3]:
transaction_data = pl.scan_csv(
    "../data/LI-Large_Trans.csv"
)

In [4]:
transaction_data.collect_schema()

Schema([('Timestamp', String),
        ('From Bank', Int64),
        ('Account', String),
        ('To Bank', Int64),
        ('Account_duplicated_0', String),
        ('Amount Received', Float64),
        ('Receiving Currency', String),
        ('Amount Paid', Float64),
        ('Payment Currency', String),
        ('Payment Format', String),
        ('Is Laundering', Int64)])

In [5]:
transaction_data.head(5).collect()

Timestamp,From Bank,Account,To Bank,Account_duplicated_0,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
str,i64,str,i64,str,f64,str,f64,str,str,i64
"""2022/08/01 00:15""",20,"""800104D70""",20,"""800104D70""",8095.07,"""US Dollar""",8095.07,"""US Dollar""","""Reinvestment""",0
"""2022/08/01 00:18""",3196,"""800107150""",3196,"""800107150""",7739.29,"""US Dollar""",7739.29,"""US Dollar""","""Reinvestment""",0
"""2022/08/01 00:23""",1208,"""80010E430""",1208,"""80010E430""",2654.22,"""US Dollar""",2654.22,"""US Dollar""","""Reinvestment""",0
"""2022/08/01 00:19""",3203,"""80010EA80""",3203,"""80010EA80""",13284.41,"""US Dollar""",13284.41,"""US Dollar""","""Reinvestment""",0
"""2022/08/01 00:27""",20,"""800104D20""",20,"""800104D20""",9.72,"""US Dollar""",9.72,"""US Dollar""","""Reinvestment""",0


In [6]:
# Number of rows
transaction_data.select(
    pl.len().alias("num_transactions")
).collect()

num_transactions
u32
176066557


In [7]:
# Check unique values for "Is Laundering"
transaction_data.select(
    "Is Laundering"
).unique().collect()

Is Laundering
i64
0
1


### Condition Data

#### Convert Timestamp from String into Datetime

In [8]:
# Create a derived column for timestamp (str -> timestamp)
transaction_data = transaction_data.with_columns(
    pl.col("Timestamp")
      .str.to_datetime("%Y/%m/%d %H:%M")
      .alias("Timestamp")
)

In [9]:
transaction_data.select(
    pl.col("Timestamp").min().alias("start_date"),
    pl.col("Timestamp").max().alias("end_date")
).collect()

start_date,end_date
datetime[μs],datetime[μs]
2022-08-01 00:00:00,2023-01-12 10:18:00


#### Reduce Dataset

In [10]:
daily_transactions = (
    transaction_data
    .group_by(
        pl.col("Timestamp").dt.date().alias("Date")
    )
    .agg(
        pl.len().alias("Transactions")
    )
    .sort("Date")
    .collect()
)

daily_transactions

Date,Transactions
date,u32
2022-08-01,4377581
2022-08-02,1894426
2022-08-03,1891657
2022-08-04,1890988
2022-08-05,2957130
2022-08-06,813056
2022-08-07,814101
2022-08-08,1891591
2022-08-09,1890247


In [ ]:
'''
    Observations:
        - After 2022-11-05, we end up only seeing laundered transactions that take multiple days to pan out (https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/discussion/427517)
                
    We filter the data for two main reason:
- For meaningful analysis and results, we need laundered transactions mixed in with normal transactions. Data after 2022-11-05 does not represent such data.

    --> Outcome: Filter to data up until 2022-11-05
'''

filtered_data = transaction_data.filter(pl.col("Timestamp") < pl.date(2022, 11, 6))

In [27]:
daily_transactions = (
    filtered_data
    .group_by(
        pl.col("Timestamp").dt.date().alias("Date")
    )
    .agg(
        pl.len().alias("Transactions")
    )
    .sort("Date")
    .collect()
)

daily_transactions

Date,Transactions
date,u32
2022-08-01,4377581
2022-08-02,1894426
2022-08-03,1891657
2022-08-04,1890988
2022-08-05,2957130
2022-08-06,813056
2022-08-07,814101
2022-08-08,1891591
2022-08-09,1890247


### Save Data

In [28]:
# Save the lazy cleaned dataset directly to CSV (streaming, low memory)
filtered_data.sink_parquet("../data/filtered_transactions.parquet")